---
# Task Guardrails
Task guardrails provide a way to validate and transform task outputs before they are passed to the next task. This feature helps ensure data quality and provides feedback to agents when their output doesn’t meet specific criteria. <br>

Guardrails are implemented as Python functions that contain custom validation logic, giving you complete control over the validation process and ensuring reliable, deterministic results.

----

In [3]:
from crewai import Agent, LLM
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file
#os.environ["OPENAI_API_KEY"] = os.getenv("GEMINI_API_KEY")


llm = LLM(model="gemini/gemini-2.5-flash", verbose=True, temperature=0.5,
          api_key=os.getenv("GOOGLE_API_KEY"))

In [4]:
def validate_summary_length(task_output):
    try:
        print("Validating summary length")
        task_str_output = str(task_output)
        total_words = len(task_str_output.split())

        print(f"Word count: {total_words}")

        if total_words > 150:
            print("Summary exceeds 150 words")
            return (False, f"Summary exceeds 150 words. Word count: {total_words}")

        if total_words == 0:
            print("Summary is empty")
            return (False, "Generated summary is empty.")

        print("Summary is valid")
        return (True, task_output)

    except Exception as e:
        print("Validation system error")
        return (False, f"Validation system error: {str(e)}")


In [6]:
from crewai import Task, Agent

summary_agent = Agent(
    role="Summary Agent",
    goal="Summarize the research paper 'Convolutional Neural Networks' in 150 words.",
    backstory="You are a specialized agent that summarizes research papers.",
    verbose=True,
    llm=llm
)

summary_task = Task(
    description="Summarize a research paper in 150 words.",
    expected_output="A concise research summary 150 words.",
    agent=summary_agent,
    guardrail=validate_summary_length,

    max_retries=3
)


In [7]:
from crewai import Crew

summary_crew = Crew(
    agents=[summary_agent],
    tasks=[summary_task],
    verbose=True
)

result = summary_crew.kickoff()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3598dcd4-688b-4310-a308-0c6863c141e5                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Agent                                                                                           │
│                                                                                                                 │
│  Task: Summarize a research paper in 150 words.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Agent                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Convolutional Neural Networks (CNNs) represent a foundational architecture in deep learning, primarily         │
│  excelling in visual imagery analysis. Research papers on CNNs typically introduce them as specialized neural   │
│  networks designed to process data with a known grid-like topology, such as images. Key components include      │
│  convolutional layers, which learn hierarchical features by applying filters to input data; pooling layers,     │
│  which reduce dimensionality and computational complexity; and activation functions, which introduce            │
│  non-linearity. These layers are often followed by fully connected layers for classification or regression.     │
│  CNNs leverage local connections, shared weights, and spatial or temporal subsampling, making them highly       │
│  efficient and effective at recognizing patterns invariant to translation. Their ability to automatically       │
│  learn robust feature representations from raw data has revolutionized fields like computer vision, enabling    │
│  breakthroughs in image classification, object detection, and segmentation, and establishing them as a          │
│  cornerstone of modern AI.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_summary_length(task_output):                                                                │
│      try:...                                                                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Validating summary length
Word count: 140
Summary is valid


╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: bbbb57bd-82e9-47f3-b067-4006fc595310                                                                     │
│  Agent: Summary Agent                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3598dcd4-688b-4310-a308-0c6863c141e5                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Convolutional Neural Networks (CNNs) represent a foundational architecture in deep learning,     │
│  primarily excelling in visual imagery analysis. Research papers on CNNs typically introduce them as            │
│  specialized neural networks designed to process data with a known grid-like topology, such as images. Key      │
│  components include convolutional layers, which learn hierarchical features by applying filters to input data;  │
│  pooling layers, which reduce dimensionality and computational complexity; and activation functions, which      │
│  introduce non-linearity. These layers are often followed by fully connected layers for classification or       │
│  regression. CNNs leverage local connections, shared weights, and spatial or temporal subsampling, making them  │
│  highly efficient and effective at recognizing patterns invariant to translation. Their ability to              │
│  automatically learn robust feature representations from raw data has revolutionized fields like computer       │
│  vision, enabling breakthroughs in image classification, object detection, and segmentation, and establishing   │
│  them as a cornerstone of modern AI.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

### Ask the agent to write a 200 word summary instead and notice the guardrail output:

In [8]:
from crewai import Task, Agent

summary_agent = Agent(
    role="Summary Agent",
    goal="Summarize the research paper 'Convolutional Neural Networks' in 200 words.",
    backstory="You are a specialized agent that summarizes research papers.",
    verbose=True,
    llm=llm
)

summary_task = Task(
    description="Summarize a research paper in 200 words.",
    expected_output="A concise research summary 200 words.",
    agent=summary_agent,
    guardrail=validate_summary_length,
    max_retries=3
)


In [9]:
from crewai import Crew

summary_crew = Crew(
    agents=[summary_agent],
    tasks=[summary_task],
    verbose=True
)

result = summary_crew.kickoff()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7270f315-7dc0-4987-9a47-8d5cddbfaf6b                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Agent                                                                                           │
│                                                                                                                 │
│  Task: Summarize a research paper in 200 words.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Agent                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Convolutional Neural Networks (CNNs) are a specialized class of deep learning models primarily designed for    │
│  analyzing visual imagery. They leverage a hierarchical structure to automatically learn spatial hierarchies    │
│  of features from input data, making them highly effective for tasks like image classification, object          │
│  detection, and segmentation.                                                                                   │
│                                                                                                                 │
│  The core components of a CNN include convolutional layers, pooling layers, and fully connected layers.         │
│  Convolutional layers apply learnable filters to the input, detecting specific features such as edges,          │
│  textures, or patterns. These filters slide across the input, performing convolutions and generating feature    │
│  maps. Pooling layers (e.g., max pooling) then reduce the dimensionality of these feature maps, decreasing      │
│  computational complexity and providing a degree of translation invariance. Non-linear activation functions,    │
│  like ReLU, are typically applied after convolutional layers.                                                   │
│                                                                                                                 │
│  Finally, one or more fully connected layers, similar to those in traditional neural networks, take the         │
│  high-level features extracted by the convolutional and pooling layers and perform the final classification or  │
│  regression. CNNs' success stems from their ability to capture local dependencies and scale invariance through  │
│  weight sharing and local receptive fields, significantly outperforming traditional methods in various          │
│  computer vision benchmarks.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_summary_length(task_output):                                                                │
│      try:...                                                                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 2dcc65c7-f0ac-4d0a-b533-d35baa2d0e71                                                                     │
│  Agent: Summary Agent                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 2dcc65c7-f0ac-4d0a-b533-d35baa2d0e71                                                                     │
│  Agent: Summary Agent                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 2dcc65c7-f0ac-4d0a-b533-d35baa2d0e71                                                                     │
│  Agent: Summary Agent                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 7270f315-7dc0-4987-9a47-8d5cddbfaf6b                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception: Task failed guardrail validation after 3 retries. Last error: Summary exceeds 150 words. Word count: 175

---

## Tasks Guardrails with Pydantic

In [10]:
from pydantic import BaseModel

class ResearchReport(BaseModel):
    """Represents a structured research report"""
    title: str
    summary: str
    key_findings: list[str]

In [12]:
import json
from typing import Tuple, Any

def validate_json_report(result: str) -> Tuple[bool, Any]:
    """Ensures AI-generated output is valid JSON with required fields."""
    try:
        # Parse JSON output
        data = json.loads(result.pydantic.model_dump_json())

        # Check required fields
        if "title" not in data or "summary" not in data or "key_findings" not in data:
            return (False, "Missing required fields: title, summary, or key_findings.")

        return (True, result)  # JSON is valid
    except json.JSONDecodeError:
        return (False, "Invalid JSON format. Please ensure correct syntax.")


In [13]:
from crewai import Agent

# Create the AI Agent
research_report_agent = Agent(
    role="Research Analyst",
    goal="Generate structured JSON reports for research papers",
    backstory="You are an expert in technical writing and structured reporting.",
    verbose=False,
    llm=llm
)


In [14]:
from crewai import Task

research_report_task = Task(
    description="Generate a structured research report in valid JSON format.",
    expected_output="A valid JSON object containing 'title', 'summary', and 'key_findings'.",
    agent=research_report_agent,
    output_pydantic=ResearchReport,  # Ensures structured output
    guardrail=validate_json_report,  # Validate output before passing to next step
    max_retries=3  # Allow up to 3 retries if validation fails
)

In [15]:
from crewai import Crew

research_crew = Crew(
    agents=[research_report_agent],
    tasks=[research_report_task],
    verbose=True  # Display execution details
)


In [16]:
result = research_crew.kickoff()

# Display the validated JSON output
print("Final Research Report:", result.pydantic)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ba1f85d4-9aac-46a9-8664-3d32a8e7389f                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: Generate a structured research report in valid JSON format.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "title": "The Impact of Remote Work on Employee Productivity and Well-being: A Longitudinal Study",          │
│    "summary": "This study investigates the multifaceted effects of remote work on employee productivity and     │
│  well-being, drawing upon a longitudinal dataset collected from 500 knowledge workers across various            │
│  industries over an 18-month period. Utilizing a mixed-methods approach, including quantitative surveys on      │
│  self-reported productivity, work-life balance, and mental health metrics, alongside qualitative interviews,    │
│  the research aims to provide a comprehensive understanding of the benefits and challenges associated with the  │
│  widespread adoption of remote work models. Findings indicate a complex relationship, with initial boosts in    │
│  perceived productivity often followed by declines in well-being if not adequately managed through              │
│  organizational support and clear boundaries.",                                                                 │
│    "key_findings": [                                                                                            │
│      "Initial increase in perceived productivity (first 6 months) attributed to reduced commute times and       │
│  increased autonomy.",                                                                                          │
│      "Significant decline in reported well-being (after 9 months) linked to blurred work-life boundaries,       │
│  increased isolation, and communication challenges.",                                                           │
│      "Organizations providing structured support (e.g., mental health resources, clear communication            │
│  protocols, virtual team-building) showed higher sustained productivity and well-being scores.",                │
│      "Individual differences in personality traits (e.g., self-discipline, introversion/extroversion)           │
│  significantly moderated the impact of remote work on both productivity and well-being.",                       │
│      "The study highlights the critical need for tailored organizational strategies to optimize remote work     │
│  benefits while mitigating its potential drawbacks."                                                            │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_json_report(result: str) -> Tuple[boo...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 33703119-5b3e-4720-83db-ab12efb993aa                                                                     │
│  Agent: Research Analyst                                                                                        │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ba1f85d4-9aac-46a9-8664-3d32a8e7389f                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "title": "The Impact of Remote Work on Employee Productivity and Well-being: A Longitudinal Study",          │
│    "summary": "This study investigates the multifaceted effects of remote work on employee productivity and     │
│  well-being, drawing upon a longitudinal dataset collected from 500 knowledge workers across various            │
│  industries over an 18-month period. Utilizing a mixed-methods approach, including quantitative surveys on      │
│  self-reported productivity, work-life balance, and mental health metrics, alongside qualitative interviews,    │
│  the research aims to provide a comprehensive understanding of the benefits and challenges associated with the  │
│  widespread adoption of remote work models. Findings indicate a complex relationship, with initial boosts in    │
│  perceived productivity often followed by declines in well-being if not adequately managed through              │
│  organizational support and clear boundaries.",                                                                 │
│    "key_findings": [                                                                                            │
│      "Initial increase in perceived productivity (first 6 months) attributed to reduced commute times and       │
│  increased autonomy.",                                                                                          │
│      "Significant decline in reported well-being (after 9 months) linked to blurred work-life boundaries,       │
│  increased isolation, and communication challenges.",                                                           │
│      "Organizations providing structured support (e.g., mental health resources, clear communication            │
│  protocols, virtual team-building) showed higher sustained productivity and well-being scores.",                │
│      "Individual differences in personality traits (e.g., self-discipline, introversion/extroversion)           │
│  significantly moderated the impact of remote work on both productivity and well-being.",                       │
│      "The study highlights the critical need for tailored organizational strategies to optimize remote work     │
│  benefits while mitigating its potential drawbacks."                                                            │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final Research Report: title='The Impact of Remote Work on Employee Productivity and Well-being: A Longitudinal Study' summary='This study investigates the multifaceted effects of remote work on employee productivity and well-being, drawing upon a longitudinal dataset collected from 500 knowledge workers across various industries over an 18-month period. Utilizing a mixed-methods approach, including quantitative surveys on self-reported productivity, work-life balance, and mental health metrics, alongside qualitative interviews, the research aims to provide a comprehensive understanding of the benefits and challenges associated with the widespread adoption of remote work models. Findings indicate a complex relationship, with initial boosts in perceived productivity often followed by declines in well-being if not adequately managed through organizational support and clear boundaries.' key_findings=['Initial increase in perceived productivity (first 6 months) attributed to reduced commut

---

## Getting Structured Consistent Outputs from Tasks

In [17]:
from pydantic import BaseModel

class ResearchFindings(BaseModel):
    """Structured research report output"""
    title: str
    key_findings: list[str]

class AnalysisSummary(BaseModel):
    """Structured summary of research findings"""
    insights: list[str]
    key_takeaways: str


In [19]:
from crewai import Agent, LLM

# AI Agents
research_agent = Agent(
    role="AI Researcher",
    goal="Find and summarize the latest AI advancements",
    backstory="You are an expert AI researcher who stays up to date with the latest innovations.",
    verbose=True,
    llm=llm
)

analysis_agent = Agent(
    role="AI Analyst",
    goal="Analyze AI research findings and extract key insights",
    backstory="You are a data analyst who extracts valuable insights from research data.",
    verbose=True,
    llm=llm
)

writer_agent = Agent(
    role="Tech Writer",
    goal="Write a well-structured blog post on AI trends",
    backstory="You are a technology writer skilled at transforming complex AI research into readable content.",
    verbose=True,
    llm=llm
)


In [20]:
from crewai import Task

# Step 1: Research Task
research_task = Task(
    description="Find and summarize the latest AI advancements",
    expected_output="A structured list of recent AI breakthroughs",
    agent=research_agent,
    output_pydantic=ResearchFindings  # Structured output
)

# Step 2: Analysis Task (References Research Task Output)
analysis_task = Task(
    description="Analyze AI research findings and extract key insights",
    expected_output="A structured summary with key takeaways",
    agent=analysis_agent,
    output_pydantic=AnalysisSummary,
    context=[research_task]  # Receives output from research_task
)

# Step 3: Blog Writing Task (References Both Research and Analysis)
blog_writing_task = Task(
    description="Write a detailed blog post about AI trends",
    expected_output="A well-structured blog post",
    agent=writer_agent,
    context=[research_task, analysis_task]  # Uses both research and analysis outputs
)


In [21]:
from crewai import Crew

ai_research_crew = Crew(
    agents=[research_agent, analysis_agent, writer_agent],
    tasks=[research_task, analysis_task, blog_writing_task],
    verbose=True
)

# Execute the workflow
result = ai_research_crew.kickoff()

# Print the final blog post
print("\n=== Generated Blog Post ===")
print(result.raw)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 57233017-f022-463d-a6cf-71371e3be88a                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Find and summarize the latest AI advancements                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"title": "Recent Breakthroughs in Artificial Intelligence", "key_findings": ["**Multimodal AI with Real-time  │
│  Interaction (e.g., OpenAI's GPT-4o)**: Introduction of highly capable multimodal models that seamlessly        │
│  integrate text, audio, and vision, enabling natural, real-time voice conversations with AI, understanding      │
│  emotional nuances, and processing live video feeds for complex reasoning and interaction.", "**Advanced        │
│  Open-Source Large Language Models (e.g., Meta's Llama 3)**: Release of new generations of open-source LLMs     │
│  demonstrating state-of-the-art performance, competitive with proprietary models, across various benchmarks,    │
│  and available in different parameter sizes for broader accessibility, research, and deployment.",              │
│  "**High-Fidelity Text-to-Video Generation (e.g., OpenAI's Sora, Luma AI's Dream Machine)**: Significant        │
│  progress in generating realistic, consistent, and coherent long-form video content from text prompts,          │
│  showcasing complex scene understanding, character consistency, and adherence to physical world dynamics.",     │
│  "**Revolutionary Molecular Structure and Interaction Prediction (e.g., DeepMind's AlphaFold 3)**: Development  │
│  of models capable of predicting the structure and interactions of a vast range of biological molecules,        │
│  including proteins, DNA, RNA, and ligands, opening new avenues for drug discovery, material science, and       │
│  fundamental biological understanding.", "**Massive Context Window for LLMs (e.g., Google DeepMind's Gemini     │
│  1.5 Pro with 1 Million Tokens)**: Expansion of context window capabilities in Large Language Models to         │
│  process extremely long inputs (e.g., entire books, hours of video or codebases), enabling deep understanding,  │
│  summarization, and reasoning over vast amounts of information in a single query.", "**Emergence of Foundation  │
│  Models for Robotics and Embodied AI**: Progress in developing general-purpose foundation models for robotics   │
│  that can learn diverse skills, adapt to new environments, and perform complex tasks through multimodal inputs  │
│  and learning from human demonstrations, moving towards more capable and general-purpose robotic systems."]}    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 9b0a0874-dca9-40e1-b309-c542c586d458                                                                     │
│  Agent: AI Researcher                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by 
ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001DF76BF3890>, 'Connection to 
telemetry.crewai.com timed out. (connect timeout=29.99999737739563)'))

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Writer                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## The AI Revolution Accelerates: Key Trends Shaping Our Future                                                │
│                                                                                                                 │
│  Artificial intelligence is no longer a futuristic concept; it's a rapidly evolving force reshaping             │
│  industries, scientific discovery, and human-computer interaction. The past year has seen an unprecedented      │
│  surge in AI capabilities, marked by breakthroughs that are not just incremental but truly transformative.      │
│  From models that can converse with us in real-time to AI systems that design new drugs, the pace of            │
│  innovation is breathtaking.                                                                                    │
│                                                                                                                 │
│  Let's dive into some of the most impactful trends currently defining the AI landscape:                         │
│                                                                                                                 │
│  ### 1. The Dawn of Truly Multimodal AI with Real-time Interaction                                              │
│                                                                                                                 │
│  One of the most significant leaps forward is the emergence of highly capable multimodal AI models. These       │
│  systems are no longer limited to processing just text, or just images; they seamlessly integrate and           │
│  understand text, audio, and vision simultaneously.                                                             │
│                                                                                                                 │
│  *   **Real-time Conversational AI**: Models like OpenAI's GPT-4o exemplify this, enabling natural, real-time   │
│  voice conversations with AI. They can understand emotional nuances in speech, process live video feeds, and    │
│  engage in complex reasoning based on diverse inputs. This paves the way for more intuitive and human-like      │
│  interactions with AI assistants and interfaces.                                                                │
│  *   **Complex Reasoning**: By combining different modalities, these AIs can perform more sophisticated tasks,  │
│  such as analyzing a live video of a soccer game while discussing its strategy, or helping a user troubleshoot  │
│  a device by both seeing and hearing the problem.                                                               │
│                                                                                                                 │
│  ### 2. The Rise of State-of-the-Art Open-Source Large Language Models (LLMs)                                   │
│                                                                                                                 │
│  The AI community is witnessing a powerful democratization of advanced AI capabilities through open-source      │
│  initiatives.                                                                                                   │
│                                                                                                                 │
│  *   **Competitive Performance**: The release of new ge

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 7d05dff2-83dd-4819-bbea-4d5323a99e4a                                                                     │
│  Agent: Tech Writer                                                                                             │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 57233017-f022-463d-a6cf-71371e3be88a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ## The AI Revolution Accelerates: Key Trends Shaping Our Future                                  │
│                                                                                                                 │
│  Artificial intelligence is no longer a futuristic concept; it's a rapidly evolving force reshaping             │
│  industries, scientific discovery, and human-computer interaction. The past year has seen an unprecedented      │
│  surge in AI capabilities, marked by breakthroughs that are not just incremental but truly transformative.      │
│  From models that can converse with us in real-time to AI systems that design new drugs, the pace of            │
│  innovation is breathtaking.                                                                                    │
│                                                                                                                 │
│  Let's dive into some of the most impactful trends currently defining the AI landscape:                         │
│                                                                                                                 │
│  ### 1. The Dawn of Truly Multimodal AI with Real-time Interaction                                              │
│                                                                                                                 │
│  One of the most significant leaps forward is the emergence of highly capable multimodal AI models. These       │
│  systems are no longer limited to processing just text, or just images; they seamlessly integrate and           │
│  understand text, audio, and vision simultaneously.                                                             │
│                                                                                                                 │
│  *   **Real-time Conversational AI**: Models like OpenAI's GPT-4o exemplify this, enabling natural, real-time   │
│  voice conversations with AI. They can understand emotional nuances in speech, process live video feeds, and    │
│  engage in complex reasoning based on diverse inputs. This paves the way for more intuitive and human-like      │
│  interactions with AI assistants and interfaces.                                                                │
│  *   **Complex Reasoning**: By combining different modalities, these AIs can perform more sophisticated tasks,  │
│  such as analyzing a live video of a soccer game while discussing its strategy, or helping a user troubleshoot  │
│  a device by both seeing and hearing the problem.                                                               │
│                                                                                                                 │
│  ### 2. The Rise of State-of-the-Art Open-Source Large Language Models (LLMs)                                   │
│                                                                                                                 │
│  The AI community is witnessing a powerful democratization of advanced AI capabilities through open-source      │
│  initiatives.                                                                                                   │
│                                                       


=== Generated Blog Post ===
## The AI Revolution Accelerates: Key Trends Shaping Our Future

Artificial intelligence is no longer a futuristic concept; it's a rapidly evolving force reshaping industries, scientific discovery, and human-computer interaction. The past year has seen an unprecedented surge in AI capabilities, marked by breakthroughs that are not just incremental but truly transformative. From models that can converse with us in real-time to AI systems that design new drugs, the pace of innovation is breathtaking.

Let's dive into some of the most impactful trends currently defining the AI landscape:

### 1. The Dawn of Truly Multimodal AI with Real-time Interaction

One of the most significant leaps forward is the emergence of highly capable multimodal AI models. These systems are no longer limited to processing just text, or just images; they seamlessly integrate and understand text, audio, and vision simultaneously.

*   **Real-time Conversational AI**: Models like Open

In [17]:
analysis_task.output

TaskOutput(description='Analyze AI research findings and extract key insights', name='Analyze AI research findings and extract key insights', expected_output='A structured summary with key takeaways', summary='Analyze AI research findings and extract key insights...', raw='{\n  "insights": [\n    "Generative AI has reached new heights in realism and complexity: OpenAI\'s Sora demonstrates groundbreaking text-to-video generation, creating high-quality, realistic, and imaginative videos up to a minute long, showcasing a deep understanding of physical world dynamics and consistent visual styles.",\n    "Massive context windows and inherent multimodality are redefining AI processing: Google DeepMind\'s Gemini 1.5 Pro features a revolutionary 1-million token context window and is inherently multimodal, capable of seamlessly processing and reasoning across text, images, audio, and video, enabling sophisticated analysis of vast and diverse inputs.",\n    "Open-source models are achieving stat

In [18]:
analysis_task.output.pydantic

AnalysisSummary(insights=["Generative AI has reached new heights in realism and complexity: OpenAI's Sora demonstrates groundbreaking text-to-video generation, creating high-quality, realistic, and imaginative videos up to a minute long, showcasing a deep understanding of physical world dynamics and consistent visual styles.", "Massive context windows and inherent multimodality are redefining AI processing: Google DeepMind's Gemini 1.5 Pro features a revolutionary 1-million token context window and is inherently multimodal, capable of seamlessly processing and reasoning across text, images, audio, and video, enabling sophisticated analysis of vast and diverse inputs.", "Open-source models are achieving state-of-the-art performance: Meta's Llama 3 sets new benchmarks for open models, outperforming many proprietary counterparts in performance, reasoning, and code generation, driven by significantly larger training datasets and an improved tokenizer.", "AI model families offer tailored in

---

## Async Execution

In [22]:
from pydantic import BaseModel

class AIResearchFindings(BaseModel):
    """Represents structured research on AI breakthroughs."""
    title: str
    key_findings: list[str]

class AIRegulationFindings(BaseModel):
    """Represents structured research on AI regulations."""
    region: str
    key_policies: list[str]

class FinalAIReport(BaseModel):
    """Combines AI research & regulation analysis into a report."""
    executive_summary: str
    key_trends: list[str]


In [23]:
from crewai import Agent

# Researcher for AI breakthroughs
research_agent = Agent(
    role="AI Researcher",
    goal="Find and summarize the latest AI breakthroughs",
    backstory="An expert AI researcher who tracks technological advancements.",
    verbose=True,
    llm=llm
)

# Analyst for AI regulations
regulation_agent = Agent(
    role="AI Policy Analyst",
    goal="Analyze global AI regulations and summarize policies",
    backstory="A government policy expert specializing in AI ethics and laws.",
    verbose=True,
    llm=llm
)

# Writer for the final AI report
writer_agent = Agent(
    role="AI Report Writer",
    goal="Write a structured report combining AI breakthroughs and regulations",
    backstory="A professional technical writer who crafts AI research reports.",
    verbose=True,
    llm=llm
)


In [24]:
from crewai import Task

# Task 1: AI Breakthroughs Research (Asynchronous)
research_ai_task = Task(
    description="Research the latest AI advancements and summarize key breakthroughs.",
    expected_output="A structured list of AI breakthroughs.",
    agent=research_agent,
    output_pydantic=AIResearchFindings,
    async_execution=True  # Runs asynchronously
)

# Task 2: AI Regulation Analysis (Asynchronous)
research_regulation_task = Task(
    description="Analyze the latest AI regulations worldwide and summarize key policies.",
    expected_output="A structured summary of AI regulations by region.",
    agent=regulation_agent,
    output_pydantic=AIRegulationFindings,
    async_execution=True  # Runs asynchronously
)

# Task 3: Generate AI Research Report (Depends on the first two tasks)
generate_report_task = Task(
    description="Write a structured report summarizing AI breakthroughs and regulations.",
    expected_output="A final AI report summarizing both aspects.",
    agent=writer_agent,
    output_pydantic=FinalAIReport,
    context=[research_ai_task, research_regulation_task]  # Waits for these tasks to complete
)


In [ ]:
from crewai import Crew

ai_research_crew = Crew(
    agents=[research_agent, regulation_agent, writer_agent],
    tasks=[research_ai_task, research_regulation_task, generate_report_task],
    verbose=True
)

# Execute the workflow
result = ai_research_crew.kickoff()

# Print the final AI report
print("\n=== Generated AI Report ===")
print(result.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e2900e3f-3ed6-4f83-b6d8-5fe391894c7e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Research the latest AI advancements and summarize key breakthroughs.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Policy Analyst                                                                                       │
│                                                                                                                 │
│  Task: Analyze the latest AI regulations worldwide and summarize key policies.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Policy Analyst                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "region": "European Union (EU)",                                                                             │
│    "key_policies": [                                                                                            │
│      "AI Act: World's first comprehensive AI law, adopting a risk-based approach (unacceptable, high, limited,  │
│  minimal risk). Prohibits certain AI systems (e.g., social scoring, real-time remote biometric identification   │
│  in public spaces by law enforcement, with narrow exceptions). Imposes strict requirements on high-risk AI      │
│  systems (e.g., conformity assessments, human oversight, data governance, transparency, cybersecurity).         │
│  Establishes an AI Office.",                                                                                    │
│      "General Data Protection Regulation (GDPR): Applies to AI systems processing personal data, emphasizing    │
│  data protection principles, individual rights (e.g., right to explanation for automated decisions), and data   │
│  minimization."                                                                                                 │
│    ]                                                                                                            │
│  },                                                                                                             │
│  {                                                                                                              │
│    "region": "United States (US)",                                                                              │
│    "key_policies": [                                                                                            │
│      "Executive Order on the Safe, Secure, and Trustworthy Development and Use of Artificial Intelligence       │
│  (October 2023): Directs federal agencies to set new standards for AI safety and security, protect privacy,     │
│  advance equity, promote innovation, and ensure responsible government use of AI. Mandates red-teaming,         │
│  watermarking, and reporting requirements for powerful AI models.",                                             │
│      "National Institute of Standards and Technology (NIST) AI Risk Management Framework (AI RMF): Voluntary    │
│  framework designed to manage risks associated with AI, promoting trustworthy AI. Focuses on govern, map,       │
│  measure, and manage functions.",                                                                               │
│      "State-level initiatives: Various states are developing their own AI regulations, such as Colorado's       │
│  proposed AI Act (similar to EU AI Act principles) and New York City's Local Law 144 (regulating automated      │
│  employment decision tools)."                                                                                   │
│    ]                                                                                                            │
│  },                                                                                                             │
│  {                                                                                                              │
│    "region": "United Kingdom (UK)",                    

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "title": "Latest AI Breakthroughs: Multimodal Models, Generative AI, and Scientific Discovery",              │
│    "key_findings": [                                                                                            │
│      "**Multimodal Large Language Models (LLMs):** Significant advancements in models like OpenAI's GPT-4o,     │
│  Google's Gemini 1.5 Pro/Flash, and Anthropic's Claude 3 family, integrating text, audio, and vision for more   │
│  natural human-computer interaction, real-time responsiveness, and massive context windows (up to 1 million     │
│  tokens).",                                                                                                     │
│      "**Breakthroughs in Generative Video AI:** OpenAI's Sora demonstrated unprecedented capabilities in        │
│  generating long, coherent, and realistic videos from text prompts, showcasing a strong understanding of        │
│  physical world dynamics. Other platforms like Luma AI and Krea AI also made strides in accessible video        │
│  generation.",                                                                                                  │
│      "**AI for Accelerated Scientific Discovery (AlphaFold 3):** Isomorphic Labs and DeepMind's AlphaFold 3     │
│  achieved a major milestone by predicting the structure and interactions of all life's molecules (proteins,     │
│  DNA, RNA, ligands), revolutionizing drug discovery, material science, and fundamental biological research.",   │
│      "**Enhanced Embodied AI and Robotics:** Progress in dexterous robot manipulation, human-robot              │
│  interaction, and the integration of LLMs for high-level planning and natural language control of robots,       │
│  leading to more versatile and capable robotic systems.",                                                       │
│      "**Open-Source LLM Advancements:** Meta AI's Llama 3 series set new benchmarks for open-source models,     │
│  offering significantly improved reasoning, code generation, and general performance across various sizes,      │
│  fostering innovation in the broader AI community.",                                                            │
│      "**Focus on AI Safety, Alignment, and Efficiency:** Increased research and development into ensuring AI    │
│  models are safe, aligned with human values, and robust against misuse. Efforts also concentrated on creating   │
│  more efficient 'small language models' (SLMs) and methods for identifying AI-generated content.",              │
│      "**Advanced 3D Content Generation:** Continued improvements in techniques like Neural Radiance Fields      │
│  (NeRFs) and Gaussian Splatting enable highly photorealistic 3D scene reconstruction and rendering from 2D      │
│  inputs, with applications in virtual reality, gaming, and digital twins."                                      │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 808c9c06-878e-42bd-a5a7-1345c37c1058                                                                     │
│  Agent: AI Researcher                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: d9b485f0-a963-434e-9bbe-d01fea9e760a                                                                     │
│  Agent: AI Policy Analyst                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception in thread Thread-33 (_execute_task_async):
Traceback (most recent call last):
  File "c:\Satish\AgenticAI2.0\AgenticAI\my-crewai-venv\Lib\site-packages\crewai\utilities\converter.py", line 166, in convert_to_model
    escaped_result = json.dumps(json.loads(result, strict=False))
                                ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Satish\miniconda3\Lib\json\__init__.py", line 359, in loads
    return cls(**kw).decode(s)
           ~~~~~~~~~~~~~~~~^^^
  File "C:\Users\Satish\miniconda3\Lib\json\decoder.py", line 348, in decode
    raise JSONDecodeError("Extra data", s, end)
json.decoder.JSONDecodeError: Extra data: line 7 column 2 (char 750)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Satish\AgenticAI2.0\AgenticAI\my-crewai-venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 1890, in completion
    response = client.post(url=url, he

---

## Callbacks

In [23]:
from crewai import Agent

# AI Researcher Agent
research_agent = Agent(
    role="AI News Researcher",
    goal="Find and summarize the latest AI news from trusted sources",
    backstory="You are a dedicated AI journalist who follows the latest advancements in artificial intelligence.",
    verbose=False,
    llm=llm
)


In [24]:
def notify_team(output):

    print(f"""Task Completed!
              Task: {output.description}
              Output Summary: {output.summary}""")

    with open("latest_ai_news.txt", "w") as f:
        f.write(f"Task: {output.description}\n")
        f.write(f"Output Summary: {output.summary}\n")
        f.write(f"Full Output: {output.raw}\n")
    
    print("News summary saved to latest_ai_news.txt")

In [25]:
from crewai import Task

research_news_task = Task(
    description="Find and summarize the latest AI breakthroughs from the last week.",
    expected_output="A structured summary of AI news headlines.",
    agent=research_agent,
    callback=notify_team  # Calls the function after task completion
)


In [26]:
from crewai import Crew

ai_news_crew = Crew(
    agents=[research_agent],
    tasks=[research_news_task],
    verbose=False
)

# Execute the workflow
result = ai_news_crew.kickoff()


Task Completed!
              Task: Find and summarize the latest AI breakthroughs from the last week.
              Output Summary: Find and summarize the latest AI breakthroughs from the last...
News summary saved to latest_ai_news.txt


---

## Hierarchical Process

In [27]:
from crewai import Agent, LLM

llm = LLM(model="gpt-4o", api_key=os.environ["OPENAI_API_KEY"])

# research a new project idea, do the research on market demand, risk, and potential return on investment.

# Define the Manager AI
manager_agent = Agent(
    role="Project Research Manager",
    goal="Oversee the project research and ensure timely, high-quality responses.",
    backstory="""An experienced project manager responsible
                 for ensuring project research.""",
    allow_delegation=True,
    verbose=True,
    llm=llm
)

# Define the Technical Support AI
market_demand_agent = Agent(
    role="Market Demand Analyst",
    goal="Write market demand content.",
    backstory="""A skilled market demand analyst who
                 writes market demand content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)

# Define the Fiction Writer AI
risk_analysis_agent = Agent(
    role="Risk Analysis Analyst",
    goal="Write risk analysis content.",
    backstory="""A risk analysis analyst who
                 writes fiction content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)

# Define the Fiction Writer AI
return_on_investment_agent = Agent(
    role="Return on Investment Analyst",
    goal="Write return on investment content.",
    backstory="""A return on investment analyst who
                writes return on investment content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)


In [28]:
from crewai import Task

manager_task = Task(
    description="""Oversee the project research on {project_title} and ensure timely, high-quality responses.""",
    expected_output="A manager-approved response ready to be sent as an article on {project_title}.",
    agent=manager_agent, 
)

market_demand_task = Task(
    description="""Analyze the market demand for the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=market_demand_agent, 
)

risk_analysis_task = Task(
    description="""Analyze the risk of the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=risk_analysis_agent, 
)

return_on_investment_task = Task(
    description="""Analyze the return on investment of the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=return_on_investment_agent, 
)

final_report_task = Task(
    description="""Review the final responses from the 
                   market demand, risk analysis, and return on investment agents
                   and create a final report.""",
    expected_output="""A comprehensive report on the project title '{project_title}'
    containing the market demand, risk analysis, and return on investment.""",
    agent=manager_agent,
)

In [29]:
from crewai import Crew, Process

project_research_crew = Crew(
    agents=[market_demand_agent, risk_analysis_agent, return_on_investment_agent],
    
    tasks=[market_demand_task, risk_analysis_task, return_on_investment_task, final_report_task],
    
    manager_agent=manager_agent,
    
    process=Process.hierarchical,
    
    verbose=True,
)

In [30]:
inputs = {"project_title": "Amazon INC AMZN"}

result = project_research_crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ca30ae6e-9745-4080-9f39-51141d1bff72                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Research Manager                                                                                │
│                                                                                                                 │
│  Task: Analyze the market demand for the project title 'Amazon INC AMZN'                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/sourangshupal/Downloads/crewai-advanced/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Demand Analyst                                                                                   │
│                                                                                                                 │
│  Task: How would you categorize the market demand for the project title 'Amazon INC AMZN'? Is it more aligned   │
│  with technical aspects or does it fall under fiction?                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Demand Analyst                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The project title 'Amazon INC AMZN' should be categorized as 'Technical' based on market demand. Amazon INC,   │
│  represented by the stock ticker AMZN, is a major multinational technology and retail company known for its     │
│  vast e-commerce platform, cloud computing services (AWS), digital streaming, and artificial intelligence. The  │
│  demand for this project is aligned with technical aspects because it encompasses the significant business      │
│  operations and technological innovations driven by Amazon in various industry sectors. This classification     │
│  also accounts for the analysis of market trends, investor interests, and the technological advancements that   │
│  Amazon continually pursues to maintain its leadership in the global market. Therefore, the market demand for   │
│  Amazon INC clearly indicates a technical orientation rather than fiction.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"task\": \"Collect and review the final responses regarding the market demand and return on investment for  │
│  Amazon INC (AMZN). Ensure that all relevant information aligns with the context of the technical aspects of    │
│  Amazon's operations.\", \"context\": \"The project focuses on Amazon INC (AMZN) categorized as 'Technical',    │
│  considering its role in e-commerce, cloud computing, digital streaming, and AI. This should cover their        │
│  business operations and technological innovations driving market demand and assess the financial returns from  │
│  these innovations.\", \"coworker\": \"Project Research Manager\"}"                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The market demand for Amazon Inc (AMZN) is shaped by its diverse operations across various sectors, including  │
│  e-commerce, cloud computing, digital streaming, and artificial intelligence (AI). Amazon's e-commerce          │
│  platform continues to dominate globally due to its user-friendly interface, vast product range, and efficient  │
│  logistics network, driving high market demand. Technological innovations like the use of AI in personalized    │
│  recommendations and efficient supply chain management enhance customer satisfaction and loyalty, further       │
│  boosting demand.                                                                                               │
│                                                                                                                 │
│  In cloud computing, Amazon Web Services (AWS) is a leader, meeting the rising demand for cloud infrastructure  │
│  fueled by digital transformation across industries. AWS's continuous innovation in scalable, secure, and       │
│  cost-effective solutions ensures sustained interest and market capture, thereby driving substantial revenue    │
│  growth.                                                                                                        │
│                                                                                                                 │
│  Digital streaming is another area where Amazon Prime Video competes with major players by leveraging original  │
│  content and integrated services with Amazon Prime, increasing subscriber numbers and retention rates. Lastly,  │
│  Amazon's investment in AI extends across its operations, from personal assistants like Alexa to autonomous     │
│  warehouse robotics, showcasing a commitment to technological advancement and improving operational             │
│  efficiency.                                                                                                    │
│                                                                                                                 │
│  From a financial perspective, Amazon's focus on technological innovation drives high returns on investment.    │
│  For instance, AWS contributes significantly to Amazon's overall profitability, counterbalancing the thin       │
│  margins in retail operations. The strategic integration of AI across business units leads to cost              │
│  efficiencies and revenue optimization, enhancing the company's financial performance.                          │
│                                                                                                                 │
│  Overall, Amazon's technical advancements cement its market position and provide compelling returns on          │
│  investment, underlining the effectiveness of their strategic innovations in responding to market needs and     │
│  financial aspirations.                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Research Manager                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Report on Amazon INC (AMZN)**                                                                  │
│                                                                                                                 │
│  **Market Demand:**                                                                                             │
│                                                                                                                 │
│  The market demand for Amazon Inc (AMZN) is shaped by its diverse operations across various sectors, including  │
│  e-commerce, cloud computing, digital streaming, and artificial intelligence (AI). Amazon's e-commerce          │
│  platform continues to dominate globally due to its user-friendly interface, vast product range, and efficient  │
│  logistics network, driving high market demand. Technological innovations like the use of AI in personalized    │
│  recommendations and efficient supply chain management enhance customer satisfaction and loyalty, further       │
│  boosting demand.                                                                                               │
│                                                                                                                 │
│  In cloud computing, Amazon Web Services (AWS) is a leader, meeting the rising demand for cloud infrastructure  │
│  fueled by digital transformation across industries. AWS's continuous innovation in scalable, secure, and       │
│  cost-effective solutions ensures sustained interest and market capture, thereby driving substantial revenue    │
│  growth.                                                                                                        │
│                                                                                                                 │
│  Digital streaming is another area where Amazon Prime Video competes with major players by leveraging original  │
│  content and integrated services with Amazon Prime, increasing subscriber numbers and retention rates. Lastly,  │
│  Amazon's investment in AI extends across its operations, from personal assistants like Alexa to autonomous     │
│  warehouse robotics, showcasing a commitment to technological advancement and improving operational             │
│  efficiency.                                                                                                    │
│                                                                                                                 │
│  **Risk Analysis:**                                                                                             │
│                                                                                                                 │
│  To analyze the risk profile of Amazon INC (AMZN) within its technical projects, we must consider the array of  │
│  sectors the company operates in—e-commerce, cloud computing, digital streaming, and artificial intelligence.   │
│  Each of these sectors presents unique risks linked to technological innovations, market competition, and       │
│  regulatory changes.                                                                                            │
│                                                                                                                 │
│  1. **Technological Innovations:**                     

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 681bd08b-c6ba-4980-9f53-519f758e027b                                                                     │
│  Agent: Project Research Manager                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ca30ae6e-9745-4080-9f39-51141d1bff72                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Comprehensive Report on Amazon INC (AMZN)**                                                    │
│                                                                                                                 │
│  **Market Demand:**                                                                                             │
│                                                                                                                 │
│  The market demand for Amazon Inc (AMZN) is shaped by its diverse operations across various sectors, including  │
│  e-commerce, cloud computing, digital streaming, and artificial intelligence (AI). Amazon's e-commerce          │
│  platform continues to dominate globally due to its user-friendly interface, vast product range, and efficient  │
│  logistics network, driving high market demand. Technological innovations like the use of AI in personalized    │
│  recommendations and efficient supply chain management enhance customer satisfaction and loyalty, further       │
│  boosting demand.                                                                                               │
│                                                                                                                 │
│  In cloud computing, Amazon Web Services (AWS) is a leader, meeting the rising demand for cloud infrastructure  │
│  fueled by digital transformation across industries. AWS's continuous innovation in scalable, secure, and       │
│  cost-effective solutions ensures sustained interest and market capture, thereby driving substantial revenue    │
│  growth.                                                                                                        │
│                                                                                                                 │
│  Digital streaming is another area where Amazon Prime Video competes with major players by leveraging original  │
│  content and integrated services with Amazon Prime, increasing subscriber numbers and retention rates. Lastly,  │
│  Amazon's investment in AI extends across its operations, from personal assistants like Alexa to autonomous     │
│  warehouse robotics, showcasing a commitment to technological advancement and improving operational             │
│  efficiency.                                                                                                    │
│                                                                                                                 │
│  **Risk Analysis:**                                                                                             │
│                                                                                                                 │
│  To analyze the risk profile of Amazon INC (AMZN) within its technical projects, we must consider the array of  │
│  sectors the company operates in—e-commerce, cloud computing, digital streaming, and artificial intelligence.   │
│  Each of these sectors presents unique risks linked to technological innovations, market competition, and       │
│  regulatory changes.                                                                                            │
│                                                       

---

In [31]:
from IPython.display import Markdown
Markdown(result.raw)

**Comprehensive Report on Amazon INC (AMZN)**

**Market Demand:**

The market demand for Amazon Inc (AMZN) is shaped by its diverse operations across various sectors, including e-commerce, cloud computing, digital streaming, and artificial intelligence (AI). Amazon's e-commerce platform continues to dominate globally due to its user-friendly interface, vast product range, and efficient logistics network, driving high market demand. Technological innovations like the use of AI in personalized recommendations and efficient supply chain management enhance customer satisfaction and loyalty, further boosting demand.

In cloud computing, Amazon Web Services (AWS) is a leader, meeting the rising demand for cloud infrastructure fueled by digital transformation across industries. AWS's continuous innovation in scalable, secure, and cost-effective solutions ensures sustained interest and market capture, thereby driving substantial revenue growth.

Digital streaming is another area where Amazon Prime Video competes with major players by leveraging original content and integrated services with Amazon Prime, increasing subscriber numbers and retention rates. Lastly, Amazon's investment in AI extends across its operations, from personal assistants like Alexa to autonomous warehouse robotics, showcasing a commitment to technological advancement and improving operational efficiency.

**Risk Analysis:**

To analyze the risk profile of Amazon INC (AMZN) within its technical projects, we must consider the array of sectors the company operates in—e-commerce, cloud computing, digital streaming, and artificial intelligence. Each of these sectors presents unique risks linked to technological innovations, market competition, and regulatory changes.

1. **Technological Innovations:**
   - **E-commerce:** As a leader in the e-commerce sector, Amazon faces the ongoing challenge of maintaining its technological edge. The risk here lies in whether the company can continue to innovate in areas like logistics, customer experience, and payment systems to stay ahead of rivals.
   - **Cloud Computing (AWS):** The cloud services sector is highly dynamic, and technological advancements occur rapidly. Amazon Web Services (AWS) must consistently evaluate and adopt cutting-edge technologies such as serverless computing and AI-driven automation to avoid obsolescence.
   - **AI and Digital Streaming:** Amazon is heavily investing in artificial intelligence, which powers recommendations and operational efficiencies. The risk is that AI technologies rapidly evolve, and being at the forefront necessitates constant innovation, which requires significant investment.

2. **Market Competition:**
   - Amazon operates in highly competitive environments. Competitors like Microsoft and Google in cloud computing, and platforms such as Netflix in digital streaming, exert pressure on Amazon to innovate and sustain market share. Intense competition can impact pricing strategies and profit margins.
   - The e-commerce space is increasingly crowded with competitors burgeoning worldwide, including Alibaba and traditional retailers transitioning online.

3. **Regulatory Changes:**
   - Amazon faces significant regulatory scrutiny from governments worldwide, particularly concerning antitrust laws, data privacy regulations, and labor policies.
   - Recent concerns are growing around data protection and digital monopoly practices, which could lead to hefty fines and mandates to modify business practices significantly.

4. **Operational Risks:**
   - Amazon's reliance on technology implies a risk from cybersecurity threats. Breaches could result in loss of customer trust, fines for data breaches, and significant costs in remediation.
   - Logistics dependencies also pose risk; any disruption can have immediate implications on fulfillment and delivery.

**Return on Investment (ROI):**

From a financial perspective, Amazon's focus on technological innovation drives high returns on investment. For instance, AWS contributes significantly to Amazon's overall profitability, counterbalancing the thin margins in retail operations. The strategic integration of AI across business units leads to cost efficiencies and revenue optimization, enhancing the company's financial performance.

Overall, Amazon's technical advancements cement its market position and provide compelling returns on investment, underlining the effectiveness of their strategic innovations in responding to market needs and financial aspirations.

In conclusion, while Amazon INC (AMZN) holds a robust position because of its diversified technological investment and market leadership, it also faces a complex risk landscape. Continuous innovation, strategic competitive positioning, and regulatory compliance are paramount to mitigating these risks ensuring sustained success.

---

## Human input

In [32]:
from crewai import Agent, LLM

llm = LLM(model="gemini/gemini-2.5-flash")

# AI Researcher Agent
researcher_agent = Agent(
    role="Senior AI Researcher",
    goal="Discover and summarize the latest trends in AI and technology.",
    backstory="An expert in AI research who tracks emerging trends and their real-world applications.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# AI Content Strategist Agent
content_strategist_agent = Agent(
    role="Tech Content Strategist",
    goal="Transform AI research insights into compelling blog content.",
    backstory="An experienced tech writer who makes AI advancements accessible to a broad audience.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [33]:
from crewai import Task

# Step 1: AI Research with Human Oversight
ai_research_task = Task(
    description=(
        "Conduct a deep analysis of AI trends in 2025. Identify key innovations, breakthroughs, and market shifts. "
        "Before finalizing, ask a human reviewer for feedback to refine the report."
    ),
    expected_output="A structured research summary covering AI advancements in 2025.",
    agent=researcher_agent,
    human_input=True  # Requires human validation before finalizing the answer
)

# Step 2: AI-Generated Blog Post with Human Review
blog_post_task = Task(
    description=(
        "Using insights from the AI Researcher, create an engaging blog post summarizing key AI advancements. "
        "Ensure the post is informative and accessible. Before finalizing, ask a human reviewer for approval."
    ),
    expected_output="A well-structured blog post about AI trends in 2025.",
    agent=content_strategist_agent,
    human_input=True  # Requires human approval before publishing
)

In [34]:
from crewai import Crew

ai_research_crew = Crew(
    agents=[researcher_agent, content_strategist_agent],  
    tasks=[ai_research_task, blog_post_task],  
    verbose=True,  
)


In [35]:
# Execute the workflow
result = ai_research_crew.kickoff()

# Display the final validated research output
print("\n=== Final AI Research Report ===")
print(result.raw)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 30d134a3-cab8-41b4-82d1-8bea82199871                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AI Researcher                                                                                    │
│                                                                                                                 │
│  Task: Conduct a deep analysis of AI trends in 2025. Identify key innovations, breakthroughs, and market        │
│  shifts. Before finalizing, ask a human reviewer for feedback to refine the report.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AI Researcher                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## AI Trends in 2025: A Deep Analysis of Innovations, Breakthroughs, and Market Shifts                         │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│                                                                                                                 │
│  2025 marks a pivotal year for Artificial Intelligence, transitioning from nascent innovation to pervasive      │
│  integration across all sectors. The dominant themes will be the maturation and hyper-specialization of         │
│  Generative AI, a renewed focus on efficiency and smaller, purpose-built Foundation Models, and significant     │
│  acceleration of AI's application in scientific discovery, particularly in medicine and materials science.      │
│  Embodied AI and robotics will see advancements in dexterity and autonomy, while the imperative for             │
│  Trustworthy AI will drive practical implementations of Explainable AI (XAI) and robust ethical governance      │
│  frameworks. Market shifts will include the widespread democratization of AI tools, a significant move towards  │
│  vertical-specific AI solutions, and a rapidly evolving regulatory landscape that seeks to balance innovation   │
│  with responsibility. The year will underscore AI's role not just as a tool, but as a fundamental               │
│  infrastructure reshaping economies, industries, and daily life.                                                │
│                                                                                                                 │
│  ### 1. Introduction: The Age of AI Operationalization                                                          │
│                                                                                                                 │
│  Building upon the groundbreaking advancements of 2023 and 2024, particularly in large language models (LLMs)   │
│  and diffusion models, 2025 will be characterized by the operationalization, refinement, and strategic          │
│  deployment of AI. The focus will shift from demonstrating capability to achieving efficiency,                  │
│  domain-specificity, and trustworthiness at scale. This period will see AI move beyond novel applications into  │
│  the core fabric of enterprise operations, product development, and scientific research, catalyzing             │
│  unprecedented productivity gains and new forms of interaction with technology.                                 │
│                                                                                                                 │
│  ### 2. Key Innovations and Breakthroughs in 2025                                                               │
│                                                                                                                 │
│  #### 2.1. Generative AI: Hyper-Specialization & Multimodality Unleashed                                        │
│                                                                                                                 │
│  *   **Controllable and Consistent Generation:** Generative AI models will evolve significantly in their        │
│  controllability. Advancements in prompt engineering, f

 ## Final Result: ## AI Trends in 2025: A Deep Analysis of Innovations, Breakthroughs, and Market Shifts

### Executive Summary

2025 marks a pivotal year for Artificial Intelligence, transitioning from nascent innovation to pervasive integration across all sectors. The dominant themes will be the maturation and hyper-specialization of Generative AI, a renewed focus on efficiency and smaller, purpose-built Foundation Models, and significant acceleration of AI's application in scientific discovery, particularly in medicine and materials science. Embodied AI and robotics will see advancements in dexterity and autonomy, while the imperative for Trustworthy AI will drive practical implementations of Explainable AI (XAI) and robust ethical governance frameworks. Market shifts will include the widespread democratization of AI tools, a significant move towards vertical-specific AI solutions, and a rapidly evolving regulatory landscape that seeks to balance innovation with responsibility. The 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1394984c-e897-45bc-9fb4-dbba89b9adc6                                                                     │
│  Agent: Senior AI Researcher                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Content Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## AI in 2025: The Year Intelligence Gets Real (and Really Specialized)                                        │
│                                                                                                                 │
│  Remember when AI was mostly about chatbots and sci-fi movies? Well, get ready, because 2025 is shaping up to   │
│  be the year AI truly steps into its own. We're moving beyond dazzling demonstrations to an era where           │
│  Artificial Intelligence becomes a fundamental, ingrained part of our daily lives, industries, and even         │
│  scientific breakthroughs. This isn't just about bigger, flashier models; it's about making AI more efficient,  │
│  trustworthy, and deeply specialized.                                                                           │
│                                                                                                                 │
│  Here's a look at the hottest AI trends set to redefine our world in 2025:                                      │
│                                                                                                                 │
│  ### 1. Generative AI Goes Pro: Precision, Multimodality & Hyper-Specialization                                 │
│                                                                                                                 │
│  Generative AI, the magic behind creating images, text, and even music from simple prompts, is getting a        │
│  serious upgrade.                                                                                               │
│                                                                                                                 │
│  *   **Beyond "Happy Accidents":** Imagine asking an AI to draft a legal document, design an engineering        │
│  blueprint, or create marketing content that perfectly matches your brand guidelines. In 2025, generative       │
│  models will be incredibly controllable and consistent, moving from impressive novelty to predictable,          │
│  production-ready tools.                                                                                        │
│  *   **True Multimodality:** Forget just text-to-image. We're talking about AI systems that understand and      │
│  generate across multiple senses simultaneously – text, image, audio, video, 3D models, and even haptic         │
│  (touch) feedback. A single prompt could describe an object and generate its 3D model, texture, and sound!      │
│  *   **Expert AI for Every Niche:** The age of the "one-size-fits-all" generalist AI is fading. Instead, we’ll  │
│  see a boom in smaller, highly efficient, and expert generative models trained on niche datasets. Think AI      │
│  specifically for designing new drug compounds, visualizing architecture, or crafting complex financial         │
│  reports.                                                                                                       │
│  *   **Synthetic Data Revolution:** Generative AI will become a powerhouse for creating high-quality synthetic  │
│  data, solving privacy concerns and data scarcity for specialized training.                                     │
│                                                                                                                 │
│  ### 2. "Small" But Mighty Foundation Models Take Cente

 ## Final Result: ## AI in 2025: The Year Intelligence Gets Real (and Really Specialized)

Remember when AI was mostly about chatbots and sci-fi movies? Well, get ready, because 2025 is shaping up to be the year AI truly steps into its own. We're moving beyond dazzling demonstrations to an era where Artificial Intelligence becomes a fundamental, ingrained part of our daily lives, industries, and even scientific breakthroughs. This isn't just about bigger, flashier models; it's about making AI more efficient, trustworthy, and deeply specialized.

Here's a look at the hottest AI trends set to redefine our world in 2025:

### 1. Generative AI Goes Pro: Precision, Multimodality & Hyper-Specialization

Generative AI, the magic behind creating images, text, and even music from simple prompts, is getting a serious upgrade.

*   **Beyond "Happy Accidents":** Imagine asking an AI to draft a legal document, design an engineering blueprint, or create marketing content that perfectly matches your 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 971e4657-d3bf-459e-8766-8d7d54caf43e                                                                     │
│  Agent: Tech Content Strategist                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 30d134a3-cab8-41b4-82d1-8bea82199871                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ## AI in 2025: The Year Intelligence Gets Real (and Really Specialized)                          │
│                                                                                                                 │
│  Remember when AI was mostly about chatbots and sci-fi movies? Well, get ready, because 2025 is shaping up to   │
│  be the year AI truly steps into its own. We're moving beyond dazzling demonstrations to an era where           │
│  Artificial Intelligence becomes a fundamental, ingrained part of our daily lives, industries, and even         │
│  scientific breakthroughs. This isn't just about bigger, flashier models; it's about making AI more efficient,  │
│  trustworthy, and deeply specialized.                                                                           │
│                                                                                                                 │
│  Here's a look at the hottest AI trends set to redefine our world in 2025:                                      │
│                                                                                                                 │
│  ### 1. Generative AI Goes Pro: Precision, Multimodality & Hyper-Specialization                                 │
│                                                                                                                 │
│  Generative AI, the magic behind creating images, text, and even music from simple prompts, is getting a        │
│  serious upgrade.                                                                                               │
│                                                                                                                 │
│  *   **Beyond "Happy Accidents":** Imagine asking an AI to draft a legal document, design an engineering        │
│  blueprint, or create marketing content that perfectly matches your brand guidelines. In 2025, generative       │
│  models will be incredibly controllable and consistent, moving from impressive novelty to predictable,          │
│  production-ready tools.                                                                                        │
│  *   **True Multimodality:** Forget just text-to-image. We're talking about AI systems that understand and      │
│  generate across multiple senses simultaneously – text, image, audio, video, 3D models, and even haptic         │
│  (touch) feedback. A single prompt could describe an object and generate its 3D model, texture, and sound!      │
│  *   **Expert AI for Every Niche:** The age of the "one-size-fits-all" generalist AI is fading. Instead, we’ll  │
│  see a boom in smaller, highly efficient, and expert generative models trained on niche datasets. Think AI      │
│  specifically for designing new drug compounds, visualizing architecture, or crafting complex financial         │
│  reports.                                                                                                       │
│  *   **Synthetic Data Revolution:** Generative AI will become a powerhouse for creating high-quality synthetic  │
│  data, solving privacy concerns and data scarcity for specialized training.                                     │
│                                                       


=== Final AI Research Report ===
## AI in 2025: The Year Intelligence Gets Real (and Really Specialized)

Remember when AI was mostly about chatbots and sci-fi movies? Well, get ready, because 2025 is shaping up to be the year AI truly steps into its own. We're moving beyond dazzling demonstrations to an era where Artificial Intelligence becomes a fundamental, ingrained part of our daily lives, industries, and even scientific breakthroughs. This isn't just about bigger, flashier models; it's about making AI more efficient, trustworthy, and deeply specialized.

Here's a look at the hottest AI trends set to redefine our world in 2025:

### 1. Generative AI Goes Pro: Precision, Multimodality & Hyper-Specialization

Generative AI, the magic behind creating images, text, and even music from simple prompts, is getting a serious upgrade.

*   **Beyond "Happy Accidents":** Imagine asking an AI to draft a legal document, design an engineering blueprint, or create marketing content that perfect

---

In [36]:
from IPython.display import Markdown
Markdown(result.raw)

## AI in 2025: The Year Intelligence Gets Real (and Really Specialized)

Remember when AI was mostly about chatbots and sci-fi movies? Well, get ready, because 2025 is shaping up to be the year AI truly steps into its own. We're moving beyond dazzling demonstrations to an era where Artificial Intelligence becomes a fundamental, ingrained part of our daily lives, industries, and even scientific breakthroughs. This isn't just about bigger, flashier models; it's about making AI more efficient, trustworthy, and deeply specialized.

Here's a look at the hottest AI trends set to redefine our world in 2025:

### 1. Generative AI Goes Pro: Precision, Multimodality & Hyper-Specialization

Generative AI, the magic behind creating images, text, and even music from simple prompts, is getting a serious upgrade.

*   **Beyond "Happy Accidents":** Imagine asking an AI to draft a legal document, design an engineering blueprint, or create marketing content that perfectly matches your brand guidelines. In 2025, generative models will be incredibly controllable and consistent, moving from impressive novelty to predictable, production-ready tools.
*   **True Multimodality:** Forget just text-to-image. We're talking about AI systems that understand and generate across multiple senses simultaneously – text, image, audio, video, 3D models, and even haptic (touch) feedback. A single prompt could describe an object and generate its 3D model, texture, and sound!
*   **Expert AI for Every Niche:** The age of the "one-size-fits-all" generalist AI is fading. Instead, we’ll see a boom in smaller, highly efficient, and expert generative models trained on niche datasets. Think AI specifically for designing new drug compounds, visualizing architecture, or crafting complex financial reports.
*   **Synthetic Data Revolution:** Generative AI will become a powerhouse for creating high-quality synthetic data, solving privacy concerns and data scarcity for specialized training.

### 2. "Small" But Mighty Foundation Models Take Center Stage

The race for bigger models isn't the only game in town. 2025 will emphasize efficiency and accessibility.

*   **Lean, Mean, AI Machines:** Expect breakthroughs in techniques that make AI models smaller, faster, and more energy-efficient without sacrificing performance. These "small" foundation models will be deployable on your local devices, bringing AI closer to you.
*   **Always Learning:** Imagine AI that updates its knowledge dynamically, without needing a full, costly retraining. Adaptive and continual learning will make AI more relevant and useful in real-time.
*   **AI for Everyone:** Advanced "Parameter-Efficient Fine-Tuning" (PEFT) methods will make it easier and cheaper for businesses and individuals to customize powerful AI models for their specific needs, democratizing access to cutting-edge AI.

### 3. AI: The Ultimate Scientific & Health Accelerant

Get ready for AI to supercharge discovery in critical fields.

*   **Drug & Materials Discovery at Warp Speed:** Building on successes like AlphaFold, AI will dramatically cut down the time it takes to find new drug candidates, optimize material properties, and design novel materials with unprecedented accuracy.
*   **Truly Personalized Medicine:** AI will integrate your genomics, health records, and wearable data to create highly individualized treatment plans, predict risks, and prevent diseases.
*   **Saving Our Planet:** AI will boost climate modeling accuracy, optimize renewable energy grids, and help design sustainable solutions for carbon capture and biodiversity.
*   **Smarter Diagnostics & Robotics in Healthcare:** AI will provide earlier, more accurate disease detection from medical images, while robots will assist in surgeries and patient rehabilitation.

### 4. Embodied AI & Robotics: Moving with Purpose

Robots aren't just for factories anymore. They're getting smarter, more adaptable, and more collaborative.

*   **General Purpose Robots:** Breakthroughs in learning will enable robots to perform a wider range of unstructured tasks in unpredictable environments, moving beyond specialized assembly lines.
*   **Smarter Cobots:** Human-robot collaboration will become seamless. AI will allow robots to understand human intent, gestures, and voice commands, making them safer and more intuitive partners in various industries.
*   **Delicate Touch:** Innovations in "soft robotics" and AI control will give robots the dexterity to handle fragile and irregularly shaped objects with precision.
*   **Autonomous Systems Mature:** While fully self-driving cars everywhere might still be a bit off, expect significant progress in Level 4 autonomy within defined areas, along with more advanced drones and autonomous industrial equipment.

### 5. Trustworthy AI & Governance: Building Confidence

As AI becomes more powerful, trust and ethics move to the forefront.

*   **Explainable AI (XAI) in Practice:** In high-stakes areas like finance and healthcare, you’ll be able to understand *why* an AI made a certain decision. XAI tools will become standard, making AI less of a black box.
*   **Safety First:** AI systems will be designed from the ground up with robustness and safety in mind, resilient to attacks and equipped with fail-safe mechanisms.
*   **Ethical AI Frameworks:** Global regulations like the EU AI Act will push companies to invest heavily in AI ethics, privacy-preserving AI, and strong internal governance to ensure responsible development.

### 6. Edge AI & Next-Gen Hardware: The Brains Go Local

AI processing is moving closer to where the data is generated, making it faster and more private.

*   **Ubiquitous Edge AI:** Your devices, smart sensors, and local servers will increasingly process AI on-site, reducing latency, enhancing privacy, and cutting down on bandwidth needs.
*   **Custom AI Chips:** Beyond general-purpose GPUs, specialized AI chips (ASICs, FPGAs) optimized for specific tasks will drive incredible gains in energy efficiency and performance.
*   **Neuromorphic Computing:** Brain-inspired chips promise ultra-low power, event-driven processing for certain types of AI, laying groundwork for future breakthroughs.

### 7. Democratization & Industry-Specific AI: Widespread Adoption

AI is no longer just for tech giants; it's becoming accessible to everyone.

*   **No-Code/Low-Code AI:** User-friendly platforms will empower domain experts and "citizen data scientists" to build and deploy AI solutions without needing extensive coding knowledge.
*   **Vertical AI Dominance:** Generic AI tools will give way to highly specialized AI-as-a-Service (AIaaS) offerings tailored for specific industries – think AI for legal tech, agriculture, retail, and education, delivering higher value.
*   **"Invisible" AI:** AI will be an embedded, expected feature within your existing software (ERP, CRM), quietly enhancing functionalities, automating tasks, and providing insights without explicit user interaction.
*   **Evolving Talent:** Expect a surge in demand for new roles like AI Ethics Officers, AI Auditors, and **Prompt Engineers** – people who are skilled at guiding AI to produce optimal results.

### The Road Ahead: Challenges and Considerations

While 2025 promises incredible advancements, challenges remain. We'll need to address the energy consumption of large AI models, tackle data scarcity for niche applications, ensure fairness and mitigate bias in AI systems, and thoughtfully navigate AI's impact on employment. The security of AI systems and the occasional "hallucinations" of generative AI will also require continuous vigilance.

### Conclusion: A Transformative Year for Humanity

2025 is set to be a truly transformative year for Artificial Intelligence. We’re witnessing AI move from a groundbreaking technology to a foundational infrastructure, deeply embedded in every facet of our lives. The focus on efficiency, specialization, and trustworthiness will define this era, accelerating scientific discovery, automating complex tasks, and reshaping how we interact with technology. As AI’s potential continues to unfold, the imperative will be to balance relentless innovation with responsible development, ethical deployment, and sustainable practices to ensure it truly benefits all of humanity.

---
*Ready for human reviewer approval.*

In [37]:
import os
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool
from dotenv import load_dotenv
load_dotenv()

llm = LLM(model="gemini/gemini-2.0-flash")

# Loading Tools
search_tool = SerperDevTool()

# Define your agents with roles, goals, tools, and additional attributes
researcher = Agent(
    role='Senior Research Analyst',
    goal='Uncover cutting-edge developments in AI and data science',
    backstory=(
        "You are a Senior Research Analyst at a leading tech think tank. "
        "Your expertise lies in identifying emerging trends and technologies in AI and data science. "
        "You have a knack for dissecting complex data and presenting actionable insights."
    ),
    verbose=True,
    allow_delegation=False,
    tools=[search_tool],
    llm=llm
)
writer = Agent(
    role='Tech Content Strategist',
    goal='Craft compelling content on tech advancements',
    backstory=(
        "You are a renowned Tech Content Strategist, known for your insightful and engaging articles on technology and innovation. "
        "With a deep understanding of the tech industry, you transform complex concepts into compelling narratives."
    ),
    verbose=True,
    allow_delegation=False,
    tools=[search_tool],
    llm=llm
)


# Create tasks for your agents
task1 = Task(
    description=(
        "Conduct a comprehensive analysis of the latest advancements in AI in 2025. "
        "Identify key trends, breakthrough technologies, and potential industry impacts. "
        "Compile your findings in a detailed report. "
        "Make sure to check with a human if the draft is good before finalizing your answer."
    ),
    expected_output='A comprehensive full report on the latest AI advancements in 2025, leave nothing out',
    agent=researcher,
    human_input=True
)

task2 = Task(
    description=(
        "Using the insights from the researcher\'s report, develop an engaging blog post that highlights the most significant AI advancements. "
        "Your post should be informative yet accessible, catering to a tech-savvy audience. "
        "Aim for a narrative that captures the essence of these breakthroughs and their implications for the future."
    ),
    expected_output='A compelling 3 paragraphs blog post formatted as markdown about the latest AI advancements in 2025',
    agent=writer,
    human_input=True
)

# Instantiate your crew with a sequential process
crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    verbose=True,

)

# Get your crew to work!
result = crew.kickoff()

print("######################")
print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a1a7c4be-eef5-46ea-baac-02f491396eaa                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Conduct a comprehensive analysis of the latest advancements in AI in 2025. Identify key trends,          │
│  breakthrough technologies, and potential industry impacts. Compile your findings in a detailed report. Make    │
│  sure to check with a human if the draft is good before finalizing your answer.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/sourangshupal/Downloads/crewai-advanced/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Thought: Okay, I need to create a comprehensive report on the latest AI advancements in 2025. This will        │
│  involve researching key trends, breakthrough technologies, and potential industry impacts. Since I am in       │
│  2024, I'll need to extrapolate based on current trends and expert predictions. I will use the search tool to   │
│  gather information on AI trends, breakthroughs, and industry forecasts for the near future. I will synthesize  │
│  this information into a detailed report. Before finalizing, I will submit a draft for human review.            │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"AI trends and predictions 2025\"}"                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Content Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## The AI Revolution of 2025: Autonomy, Efficiency, and Accessibility                                          │
│                                                                                                                 │
│  2025 is shaping up to be a landmark year for Artificial Intelligence, marked by breakthroughs that are not     │
│  just incremental improvements, but fundamental shifts in how AI is developed and deployed. One of the most     │
│  significant trends is the rise of **Agentic AI**. These AI-powered agents are designed to operate              │
│  autonomously, making decisions and performing complex tasks with minimal human oversight. Imagine AI systems   │
│  capable of managing intricate workflows, from customer service interactions to optimizing logistics, all with  │
│  limited supervision. This move towards greater autonomy promises to unlock unprecedented levels of efficiency  │
│  and productivity across industries.                                                                            │
│                                                                                                                 │
│  Another key advancement lies in the development of **AI-specific hardware**, especially application-specific   │
│  semiconductors. These specialized chips are engineered to accelerate AI workloads, delivering superior         │
│  performance and energy efficiency. Simultaneously, **Small Language Models (SLMs)** are emerging as a          │
│  transformative technology. SLMs offer a sweet spot between performance and computational cost, enabling        │
│  broader accessibility and deployment of AI on edge devices. Furthermore, advancements in memory are leading    │
│  to near-infinite memory capacity for AI systems, facilitating improved learning and decision-making            │
│  capabilities, while optimization of inference time compute ensures faster and more efficient AI deployments.   │
│                                                                                                                 │
│  The impact of these advancements is already being felt across numerous sectors. From streamlining operations   │
│  and driving smarter strategies in business, as emphasized by PwC, to scaling AI systems and demonstrating      │
│  real gains in healthcare, as noted by McKinsey. AI is reshaping the job market, increasing productivity, and   │
│  creating demand for new skills and expertise, particularly in sectors like financial services. As AI becomes   │
│  more deeply integrated into our lives, it's crucial to address the challenges of infrastructure integration,   │
│  data privacy, and ensure responsible deployment to maximize the benefits of this powerful technology. The AI   │
│  revolution is not just coming; it's here, and it's transforming the world around us.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 ## Final Result: ## The AI Revolution of 2025: Autonomy, Efficiency, and Accessibility

2025 is shaping up to be a landmark year for Artificial Intelligence, marked by breakthroughs that are not just incremental improvements, but fundamental shifts in how AI is developed and deployed. One of the most significant trends is the rise of **Agentic AI**. These AI-powered agents are designed to operate autonomously, making decisions and performing complex tasks with minimal human oversight. Imagine AI systems capable of managing intricate workflows, from customer service interactions to optimizing logistics, all with limited supervision. This move towards greater autonomy promises to unlock unprecedented levels of efficiency and productivity across industries.

Another key advancement lies in the development of **AI-specific hardware**, especially application-specific semiconductors. These specialized chips are engineered to accelerate AI workloads, delivering superior performance and energ

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 63dd700a-31cd-417d-aa83-7003119b1e0e                                                                     │
│  Agent: Tech Content Strategist                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a1a7c4be-eef5-46ea-baac-02f491396eaa                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ## The AI Revolution of 2025: Autonomy, Efficiency, and Accessibility                            │
│                                                                                                                 │
│  2025 is shaping up to be a landmark year for Artificial Intelligence, marked by breakthroughs that are not     │
│  just incremental improvements, but fundamental shifts in how AI is developed and deployed. One of the most     │
│  significant trends is the rise of **Agentic AI**. These AI-powered agents are designed to operate              │
│  autonomously, making decisions and performing complex tasks with minimal human oversight. Imagine AI systems   │
│  capable of managing intricate workflows, from customer service interactions to optimizing logistics, all with  │
│  limited supervision. This move towards greater autonomy promises to unlock unprecedented levels of efficiency  │
│  and productivity across industries.                                                                            │
│                                                                                                                 │
│  Another key advancement lies in the development of **AI-specific hardware**, especially application-specific   │
│  semiconductors. These specialized chips are engineered to accelerate AI workloads, delivering superior         │
│  performance and energy efficiency. Simultaneously, **Small Language Models (SLMs)** are emerging as a          │
│  transformative technology. SLMs offer a sweet spot between performance and computational cost, enabling        │
│  broader accessibility and deployment of AI on edge devices. Furthermore, advancements in memory are leading    │
│  to near-infinite memory capacity for AI systems, facilitating improved learning and decision-making            │
│  capabilities, while optimization of inference time compute ensures faster and more efficient AI deployments.   │
│                                                                                                                 │
│  The impact of these advancements is already being felt across numerous sectors. From streamlining operations   │
│  and driving smarter strategies in business, as emphasized by PwC, to scaling AI systems and demonstrating      │
│  real gains in healthcare, as noted by McKinsey. AI is reshaping the job market, increasing productivity, and   │
│  creating demand for new skills and expertise, particularly in sectors like financial services. As AI becomes   │
│  more deeply integrated into our lives, it's crucial to address the challenges of infrastructure integration,   │
│  data privacy, and ensure responsible deployment to maximize the benefits of this powerful technology. The AI   │
│  revolution is not just coming; it's here, and it's transforming the world around us.                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

######################
## The AI Revolution of 2025: Autonomy, Efficiency, and Accessibility

2025 is shaping up to be a landmark year for Artificial Intelligence, marked by breakthroughs that are not just incremental improvements, but fundamental shifts in how AI is developed and deployed. One of the most significant trends is the rise of **Agentic AI**. These AI-powered agents are designed to operate autonomously, making decisions and performing complex tasks with minimal human oversight. Imagine AI systems capable of managing intricate workflows, from customer service interactions to optimizing logistics, all with limited supervision. This move towards greater autonomy promises to unlock unprecedented levels of efficiency and productivity across industries.

Another key advancement lies in the development of **AI-specific hardware**, especially application-specific semiconductors. These specialized chips are engineered to accelerate AI workloads, delivering superior performance and 